[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Cross-Validation &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Cross-validate `LinearRegression` on `X` and `y` with `KFold(10, shuffle=True, random_state=0)`, and print the ten scores, their mean and their spread, rounded to three places.

In [2]:
X = network[FEATURES]
y = network["jan_mean"]
scores = cross_val_score(LinearRegression(), X, y, cv=KFold(10, shuffle=True, random_state=0))

print(scores.round(3))
print(scores.mean().round(3), scores.std().round(3))

[0.918 0.876 0.875 0.86  0.875 0.856 0.867 0.879 0.889 0.889]
0.879 0.017


Ten folds of 120 stations each. Smaller test folds make each score noisier, so the spread is larger than with five.

**2.** Count the snow stations in each test fold of `KFold(5, shuffle=True, random_state=0)` and of `StratifiedKFold(5, shuffle=True, random_state=0)`, and print both lists.

In [3]:
snow = network["snow_on_may_1"]

plain = KFold(5, shuffle=True, random_state=0)
stratified = StratifiedKFold(5, shuffle=True, random_state=0)

print([int(snow.iloc[test].sum()) for _, test in plain.split(X)])
print([int(snow.iloc[test].sum()) for _, test in stratified.split(X, snow)])

[98, 85, 85, 100, 97]
[93, 93, 93, 93, 93]


Snow is common, at 39 percent of stations, so even plain folds hold between 85 and 100 snow stations. Stratified folds hold exactly the same number in every fold.

**3.** Score `RandomForestRegressor(random_state=0)` and `LinearRegression` with `scoring="neg_mean_absolute_error"` on the same shuffled five folds, and print each model's mean error in degrees as a positive number.

In [4]:
folds = KFold(5, shuffle=True, random_state=0)

for name, model in [("linear", LinearRegression()), ("forest", RandomForestRegressor(random_state=0))]:
    errors = cross_val_score(model, X, y, cv=folds, scoring="neg_mean_absolute_error")
    print(name, (-errors).mean().round(3))

linear 0.814
forest 0.787


Negating the scores turns them back into errors. The forest's is smaller by 0.027 degrees on the same folds, and as with R² in the worked example, the fold-by-fold differences are what say whether a gap that size is more than luck.

**4.** Sort `network` by `elevation_m`, then cross-validate `LinearRegression` on it with `cv=5` and with `KFold(5, shuffle=True, random_state=0)`, and print both sets of scores.

In [5]:
by_height = network.sort_values("elevation_m").reset_index(drop=True)

print(cross_val_score(LinearRegression(), by_height[FEATURES], by_height["jan_mean"], cv=5).round(3))
print(cross_val_score(LinearRegression(), by_height[FEATURES], by_height["jan_mean"],
                      cv=KFold(5, shuffle=True, random_state=0)).round(3))

[0.803 0.858 0.856 0.832 0.806]
[0.887 0.894 0.873 0.855 0.882]


Sorted by height, each fold in order tests one band of elevations, and the scores drop. Height is only one of the things January depends on, so the drop is smaller than for a table sorted by the January mean itself.

---

&#8592; **Back to:** [Cross-Validation](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/06-cross-validation.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
